In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./data/raw")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
USE_GPU     = False  # 채점 서버는 CPU 전용 환경 -> 반드시 False로 사전 검증
# ==================

## 1. 라이브러리 & 모듈 임포트

In [ ]:
import glob
from pathlib import Path
import pandas as pd

import sys
sys.path.append(".")
from src import preprocess, ocr_engine, postprocess

## 2. 이미지 목록 로드

In [ ]:
IMAGE_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = []
for ext in IMAGE_EXTS:
    image_paths.extend(glob.glob(str(Path(INPUT_DIR) / ext)))
image_paths = sorted(set(image_paths))
print(f"총 {len(image_paths)}장 로드")

## 3. OCR 엔진 초기화

In [ ]:
engine = ocr_engine.PaddleEngine(max_side=640)

## 4. 추론 파이프라인 (Detection+Recognition -> 후처리)

베이스라인 버전: 인식된 텍스트 중 첫 번째로 발견된 유효한 날짜를 소비기한으로 채택.
정확도를 높이려면 `src/postprocess.py`의 판별 로직(앵커 키워드/위치 기반)을 고도화할 것.

In [ ]:
import time

rows = []
start_time = time.time()

for idx, path in enumerate(image_paths, 1):
    image_id = Path(path).stem
    boxes = engine.read(path)
    year, month, day, final_date = postprocess.extract_expiry_date(boxes)

    rows.append({
        "image_id": image_id,
        "year": year,
        "month": month,
        "day": day,
        "final_date": final_date,
    })

    if idx % 50 == 0 or idx == len(image_paths):
        print(f"  [{idx}/{len(image_paths)}] 진행 중... ({time.time() - start_time:.1f}초)")

print(f"\n추론 완료 (총 {time.time() - start_time:.1f}초)")

## 5. 제출 파일 저장

스키마: `image_id, year, month, day, final_date` (미인식 시 "NONE") — 반드시 `index=False`

In [ ]:
df = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
print(f"저장 완료: {OUTPUT_PATH}")